## Read the Trained Weight and Run FP32 Inference on Zynq CPU

#### How to install library in PYNQ Kria KV260

sudo /usr/local/share/pynq-venv/bin/pip3 install torch --index-url https://download.pytorch.org/whl/cpu \
sudo /usr/local/share/pynq-venv/bin/pip3 install pandas \
sudo /usr/local/share/pynq-venv/bin/pip3 install scikit-learn

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
###########################################################################
# Dataset
###########################################################################

data = {
    'Name': [
        'Nasi Tim Ayam',
        'Telur Balado',
        'Sayur Bening Bayam',
        'Ayam Goreng Mentega',
        'Tumis Buncis Wortel',
        'Opor Ayam'
    ],
    'Sour': [1, 2, 2, 2, 1, 1],
    'Sweet': [6, 4, 3, 8, 4, 5],
    'Salty': [5, 6, 4, 6, 4, 5],
    'Spicy': [1, 7, 1, 2, 2, 2],
    # Target: 1 for Like, 0 for Dislike
    'Label': [1, 0, 0, 1, 0, 1]
}
df = pd.DataFrame(data)

# Features (Sour, Sweet, Salty, Spicy) -> 4 input dimensions
X = df[['Sour', 'Sweet', 'Salty', 'Spicy']].values
y = df['Label'].values

# Standardize feature values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # Shape: (N, 1)

In [3]:
###########################################################################
# Re-define the model architecture (must match the saved model)
###########################################################################

class FoodPreferenceANN(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8):
        super(FoodPreferenceANN, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

In [4]:
###########################################################################
# Load trained weights from .pt File
###########################################################################

model = FoodPreferenceANN(input_dim=4, hidden_dim=8)
model.load_state_dict(torch.load('/home/ubuntu/workspace/ai_food_weights.pt'))
model.eval()  # Set model to evaluation mode

print("Model weights successfully loaded from ai_food_weights.pt!")

Model weights successfully loaded from ai_food_weights.pt!


In [5]:
###########################################################################
# Perform inference using the original FP32 model
###########################################################################

model.eval()

with torch.no_grad():
    raw_logits_fp32 = model(X_tensor)
    probabilities_fp32 = torch.sigmoid(raw_logits_fp32)
    predictions_fp32 = (probabilities_fp32 >= 0.5).float()

print("\n--- FP32 Model Predictions ---")

for name, prob, pred in zip(
    df['Name'],
    probabilities_fp32,
    predictions_fp32
):
    taste = "Like" if pred.item() else "Dislike"

    print(
        f"Food: {name:22s} | "
        f"Prob(Like): {prob.item():.4f} | "
        f"Predicted: {taste}"
    )

# --- Calculate Accuracy ---
correct = (predictions_fp32 == y_tensor).sum().item()
total = y_tensor.size(0)
accuracy = correct / total * 100

print(f"\nFP32 Model Accuracy: {accuracy:.2f}% ({correct}/{total} correct)")


--- FP32 Model Predictions ---
Food: Nasi Tim Ayam          | Prob(Like): 0.9892 | Predicted: Like
Food: Telur Balado           | Prob(Like): 0.0067 | Predicted: Dislike
Food: Sayur Bening Bayam     | Prob(Like): 0.0003 | Predicted: Dislike
Food: Ayam Goreng Mentega    | Prob(Like): 0.9961 | Predicted: Like
Food: Tumis Buncis Wortel    | Prob(Like): 0.0387 | Predicted: Dislike
Food: Opor Ayam              | Prob(Like): 0.9408 | Predicted: Like

FP32 Model Accuracy: 100.00% (6/6 correct)


## Quantization and Run INT8 Inference on FPGA

In [6]:
from pynq import Overlay
from pynq import allocate
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import copy

In [7]:
###########################################################################
# FPGA systolic matrix multiplication wrapper functions
###########################################################################

def init_fpga(bitstream_path='/home/ubuntu/workspace/systolic_mmult.bit'):
    global overlay, dma, dma_send, dma_recv
    overlay = Overlay(bitstream_path)
    dma = overlay.axi_dma_0
    dma_send = dma.sendchannel
    dma_recv = dma.recvchannel

def pack_matrix(mat):
    """
    Packs a 4x4 matrix of INT8 values into a 1D array of 4 uint32 words.
    Each row (4 x 8-bit elements) is packed as: [elem3 << 24 | elem2 << 16 | elem1 << 8 | elem0]
    """
    # Cast to uint32 to ensure bitwise operations don't overflow signed representation
    mat_u32 = np.asarray(mat, dtype=np.uint8).astype(np.uint32)
    
    shifts = np.array([0, 8, 16, 24], dtype=np.uint32)
    # Shift columns and bitwise-OR along axis 1 (row-wise)
    return np.bitwise_or.reduce(mat_u32 << shifts, axis=1)

def unpack_matrix(buf):
    """
    Unpacks a 1D buffer of 8 uint64 words into a 4x4 matrix of INT32 accumulators.
    Each 64-bit word contains two 32-bit signed values:
      - Low 32 bits (bits 0..31)
      - High 32 bits (bits 32..63)
    8 words x 2 values = 16 elements (4x4 matrix).
    """
    buf64 = np.asarray(buf, dtype=np.uint64)
    
    # Extract lower 32 bits and upper 32 bits from each 64-bit word
    low_32  = (buf64 & 0xFFFFFFFF).astype(np.uint32).view(np.int32)
    high_32 = ((buf64 >> 32) & 0xFFFFFFFF).astype(np.uint32).view(np.int32)
    
    # Interleave low and high 32-bit words to reconstruct the 16 elements
    c_flat = np.empty(16, dtype=np.int32)
    c_flat[0::2] = low_32
    c_flat[1::2] = high_32
    
    return c_flat.reshape(4, 4)

def systolic_mmult(a, b):
    """
    Executes 4x4 matrix multiplication A @ B on hardware.
    - Inputs: a [4x4], b [4x4] (INT8)
    - Output: c [4x4] (INT32)
    """
    # 1. Allocate contiguous memory for DMA
    inp_buf = allocate(shape=(8,), dtype=np.uint32)  # 4 words B + 4 words A
    out_buf = allocate(shape=(8,), dtype=np.uint64)  # 8 words (16 x 32-bit outputs)
    
    # 2. Pack matrices: B first, then A (matches your working input_buffer)
    packed_b = pack_matrix(b)
    packed_a = pack_matrix(a)
    inp_buf[:4] = packed_b
    inp_buf[4:] = packed_a
    
    # 3. DMA Transfer
    dma_send.transfer(inp_buf)
    dma_recv.transfer(out_buf)
    dma_send.wait()
    dma_recv.wait()
    
    # 4. Unpack 64-bit output buffer into 4x4 INT32 matrix
    c = unpack_matrix(out_buf)
    
    # 5. Free DMA memory buffers
    del inp_buf, out_buf
    
    return c

def block_matmul(a, b, block_size=4):
    # Get the dimensions of the input matrices
    m, n_a = a.shape
    n_b, p = b.shape
    if n_a != n_b:
        raise ValueError("Matrix dimensions must be compatible for multiplication")

    # Padding matrix a
    num_row_pad_a = int(np.ceil(m / block_size) * block_size - m)
    num_col_pad_a = int(np.ceil(n_a / block_size) * block_size - n_a)
    padded_row_a = np.vstack([a, np.zeros((num_row_pad_a, a.shape[1]))])
    padded_a = np.hstack([padded_row_a, np.zeros((padded_row_a.shape[0], num_col_pad_a))])

    # Padding matrix b
    num_row_pad_b = int(np.ceil(n_b / block_size) * block_size - n_b)
    num_col_pad_b = int(np.ceil(p / block_size) * block_size - p)
    padded_row_b = np.vstack([b, np.zeros((num_row_pad_b, b.shape[1]))])
    padded_b = np.hstack([padded_row_b, np.zeros((padded_row_b.shape[0], num_col_pad_b))])
    
    # Result matrix c with padding
    c = np.zeros((padded_a.shape[0], padded_b.shape[1]))

    # Iterate over blocks (using padded matrices!)
    for i in range(0, padded_a.shape[0], block_size):
        for j in range(0, padded_b.shape[1], block_size):
            for k in range(0, padded_a.shape[1], block_size):
                A_block = padded_a[i:i+block_size, k:k+block_size]
                B_block = padded_b[k:k+block_size, j:j+block_size]

                # Multiply the blocks
                C_block = systolic_mmult(A_block.astype(np.int8), B_block.astype(np.int8))

                # Accumulate the result
                c[i:i+block_size, j:j+block_size] += C_block

    # Remove padding
    return c[:m, :p]

In [8]:
###########################################################################
# Program FPGA
###########################################################################

init_fpga()

In [9]:
###########################################################################
# Function to get quantization scales
###########################################################################

def get_linear_quantization_scales(model, calibration_loader):
    model.eval()

    # Track activation ranges entering each Linear layer
    max_input_layer1 = 0.0
    max_input_layer2 = 0.0

    with torch.no_grad():
        for batch in calibration_loader:

            # Unpack input tensor from DataLoader
            x = batch[0] if isinstance(batch, (list, tuple)) else batch

            # ---------------------------------------------------------
            # Input to Linear 1
            # ---------------------------------------------------------
            max_input_layer1 = max(
                max_input_layer1,
                torch.max(torch.abs(x)).item()
            )

            # Floating-point Linear 1
            layer1_out = model.layer1(x)

            # Floating-point ReLU
            relu_out = model.relu(layer1_out)

            # ---------------------------------------------------------
            # Input to Linear 2
            # ---------------------------------------------------------
            max_input_layer2 = max(
                max_input_layer2,
                torch.max(torch.abs(relu_out)).item()
            )

    # ---------------------------------------------------------
    # Activation quantization scales
    # ---------------------------------------------------------
    input_scale_layer1 = max_input_layer1 / 127.0
    input_scale_layer2 = max_input_layer2 / 127.0

    # ---------------------------------------------------------
    # Weight quantization scales
    # ---------------------------------------------------------
    weight_scale_layer1 = (
        torch.max(torch.abs(model.layer1.weight)).item() / 127.0
    )

    weight_scale_layer2 = (
        torch.max(torch.abs(model.layer2.weight)).item() / 127.0
    )

    return {
        "layer1_input_scale": input_scale_layer1,
        "layer1_weight_scale": weight_scale_layer1,
        "layer2_input_scale": input_scale_layer2,
        "layer2_weight_scale": weight_scale_layer2,
    }

In [10]:
###########################################################################
# Get quantization scale
###########################################################################

# Calibration dataset uses only the input features
calibration_dataset = TensorDataset(X_tensor)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=2,
    shuffle=False
)

# Calculate INT8 activation and weight scales
scales = get_linear_quantization_scales(
    model,
    calibration_loader
)

print(scales)

{'layer1_input_scale': 0.01718756345313365, 'layer1_weight_scale': 0.009734624952781858, 'layer2_input_scale': 0.031243703496737742, 'layer2_weight_scale': 0.007930493730259693}


In [11]:
###########################################################################
# FPGA Quantized INT8 linear class
###########################################################################

class FPGAQuantizedLinear(nn.Module):

    def __init__(self, original_layer, input_scale, weight_scale):
        super().__init__()

        self.in_features = original_layer.in_features
        self.out_features = original_layer.out_features

        # Quantization scales
        self.input_scale = input_scale
        self.weight_scale = weight_scale

        # ---------------------------------------------------------------
        # Quantize weights using the calibrated weight scale
        # ---------------------------------------------------------------
        weights_float = original_layer.weight.detach().cpu().numpy()

        self.weights_int8 = np.round(
            weights_float / self.weight_scale
        ).clip(-128, 127).astype(np.int8)

        # Keep bias in floating point for simplicity
        if original_layer.bias is not None:
            self.bias = original_layer.bias.detach().clone()
        else:
            self.bias = None

    def forward(self, x):

        # FP32 -> INT8
        x_np = x.detach().cpu().numpy()

        x_int8 = np.round(
            x_np / self.input_scale
        ).clip(-128, 127).astype(np.int8)

        # INT8 x INT8 -> INT32
        x1_shape = x_int8.shape
        x2_shape = self.weights_int8.T.shape
        print(f"DEBUG: FPGA matmul X1{x1_shape} * X2{x2_shape}")
        
        # Matrix multiplication with FPGA
        out_int32 = block_matmul(
            x_int8.astype(np.int32),
            self.weights_int8.T.astype(np.int32)
        )

        # INT32 -> FP32
        total_scale = self.input_scale * self.weight_scale

        out_float = (
            out_int32.astype(np.float32) * total_scale
        )

        # Add FP32 bias
        if self.bias is not None:
            out_float += self.bias.cpu().numpy()

        return torch.from_numpy(out_float).to(x.device)

In [12]:
###########################################################################
# Function to integrate quantized linear class
###########################################################################

def integrate_quantization_layer(model, scales):
    """
    Recursively replaces all nn.Linear layers with NumpyQuantizedLinear.

    Each Linear layer uses its corresponding calibrated input
    and weight quantization scales.
    """

    for name, module in model.named_children():

        if isinstance(module, nn.Linear):

            # ---------------------------------------------------------
            # Select quantization scales for this layer
            # ---------------------------------------------------------
            if name == "layer1":
                input_scale = scales["layer1_input_scale"]
                weight_scale = scales["layer1_weight_scale"]

            elif name == "layer2":
                input_scale = scales["layer2_input_scale"]
                weight_scale = scales["layer2_weight_scale"]

            else:
                raise ValueError(
                    f"No quantization scale found for layer: {name}"
                )

            # ---------------------------------------------------------
            # Create quantized layer
            # ---------------------------------------------------------
            quantized_layer = FPGAQuantizedLinear(
                module,
                input_scale,
                weight_scale
            )

            # ---------------------------------------------------------
            # Replace FP32 Linear with quantized Linear
            # ---------------------------------------------------------
            setattr(model, name, quantized_layer)

            print(
                f"Replaced layer: {name} "
                f"with NumpyQuantizedLinear "
                f"(input_scale={input_scale:.6f}, "
                f"weight_scale={weight_scale:.6f})"
            )

        else:
            # Recursively process child modules
            integrate_quantization_layer(module, scales)

In [13]:
###########################################################################
# Replace the FP32 linear class with INT8 linear class
###########################################################################

# Create an isolated deep copy for quantization
# The original FP32 model remains unchanged.
model_quantized = copy.deepcopy(model)

print("\n--- Before Quantization ---")
print("Original model:")
print("  Layer 1:", model.layer1)
print("  Layer 2:", model.layer2)

print("\nQuantized model copy:")
print("  Layer 1:", model_quantized.layer1)
print("  Layer 2:", model_quantized.layer2)
print("")

# Apply quantization ONLY to the copied model
integrate_quantization_layer(model_quantized, scales)

print("\n--- After Quantization ---")
print("Original model (unchanged):")
print("  Layer 1:", model.layer1)
print("  Layer 2:", model.layer2)

print("\nQuantized model:")
print("  Layer 1:", model_quantized.layer1)
print("  Layer 2:", model_quantized.layer2)


--- Before Quantization ---
Original model:
  Layer 1: Linear(in_features=4, out_features=8, bias=True)
  Layer 2: Linear(in_features=8, out_features=1, bias=True)

Quantized model copy:
  Layer 1: Linear(in_features=4, out_features=8, bias=True)
  Layer 2: Linear(in_features=8, out_features=1, bias=True)

Replaced layer: layer1 with NumpyQuantizedLinear (input_scale=0.017188, weight_scale=0.009735)
Replaced layer: layer2 with NumpyQuantizedLinear (input_scale=0.031244, weight_scale=0.007930)

--- After Quantization ---
Original model (unchanged):
  Layer 1: Linear(in_features=4, out_features=8, bias=True)
  Layer 2: Linear(in_features=8, out_features=1, bias=True)

Quantized model:
  Layer 1: FPGAQuantizedLinear()
  Layer 2: FPGAQuantizedLinear()


In [14]:
###########################################################################
# Perform inference using the INT8 quantized model
###########################################################################

model_quantized.eval()

with torch.no_grad():
    raw_logits_int8 = model_quantized(X_tensor)
    probabilities_int8 = torch.sigmoid(raw_logits_int8)
    predictions_int8 = (probabilities_int8 >= 0.5).float()

print("\n--- FPGA INT8 Model Predictions ---")

for name, prob, pred in zip(
    df['Name'],
    probabilities_int8,
    predictions_int8
):
    taste = "Like" if pred.item() else "Dislike"

    print(
        f"Food: {name:22s} | "
        f"Prob(Like): {prob.item():.4f} | "
        f"Predicted: {taste}"
    )

# --- Calculate Accuracy ---
correct = (predictions_int8 == y_tensor).sum().item()
total = y_tensor.size(0)
accuracy = correct / total * 100

print(
    f"\nFPGA INT8 Model Accuracy: "
    f"{accuracy:.2f}% ({correct}/{total} correct)"
)

DEBUG: FPGA matmul X1(6, 4) * X2(4, 8)
DEBUG: FPGA matmul X1(6, 8) * X2(8, 1)

--- FPGA INT8 Model Predictions ---
Food: Nasi Tim Ayam          | Prob(Like): 0.9891 | Predicted: Like
Food: Telur Balado           | Prob(Like): 0.0067 | Predicted: Dislike
Food: Sayur Bening Bayam     | Prob(Like): 0.0003 | Predicted: Dislike
Food: Ayam Goreng Mentega    | Prob(Like): 0.9960 | Predicted: Like
Food: Tumis Buncis Wortel    | Prob(Like): 0.0382 | Predicted: Dislike
Food: Opor Ayam              | Prob(Like): 0.9406 | Predicted: Like

FPGA INT8 Model Accuracy: 100.00% (6/6 correct)


In [15]:
###########################################################################
# Quantization error calculation
###########################################################################

model_quantized.eval()

with torch.no_grad():
    raw_logits_int8 = model_quantized(X_tensor)
    probabilities_int8 = torch.sigmoid(raw_logits_int8)
    predictions_int8 = (probabilities_int8 >= 0.5).float()

errors = torch.abs(probabilities_fp32 - probabilities_int8)

print("\n--- FP32 vs FPGA INT8 Model ---")
print(
    f"{'Food':22s} | "
    f"{'FP32 Prob':>10s} | "
    f"{'INT8 Prob':>10s} | "
    f"{'Abs Error':>10s} | "
    f"{'FP32 Pred':>10s} | "
    f"{'INT8 Pred':>10s}"
)
print("-" * 90)

for name, p_fp32, p_int8, error, pred_fp32, pred_int8 in zip(
    df['Name'],
    probabilities_fp32,
    probabilities_int8,
    errors,
    predictions_fp32,
    predictions_int8
):
    taste_fp32 = "Like" if pred_fp32.item() else "Dislike"
    taste_int8 = "Like" if pred_int8.item() else "Dislike"

    print(
        f"{name:22s} | "
        f"{p_fp32.item():10.4f} | "
        f"{p_int8.item():10.4f} | "
        f"{error.item():10.4f} | "
        f"{taste_fp32:>10s} | "
        f"{taste_int8:>10s}"
    )

print("-" * 90)
print(f"Mean Absolute Error:    {errors.mean().item():.6f}")
print(f"Maximum Absolute Error: {errors.max().item():.6f}")

DEBUG: FPGA matmul X1(6, 4) * X2(4, 8)
DEBUG: FPGA matmul X1(6, 8) * X2(8, 1)

--- FP32 vs FPGA INT8 Model ---
Food                   |  FP32 Prob |  INT8 Prob |  Abs Error |  FP32 Pred |  INT8 Pred
------------------------------------------------------------------------------------------
Nasi Tim Ayam          |     0.9892 |     0.9891 |     0.0001 |       Like |       Like
Telur Balado           |     0.0067 |     0.0067 |     0.0000 |    Dislike |    Dislike
Sayur Bening Bayam     |     0.0003 |     0.0003 |     0.0000 |    Dislike |    Dislike
Ayam Goreng Mentega    |     0.9961 |     0.9960 |     0.0000 |       Like |       Like
Tumis Buncis Wortel    |     0.0387 |     0.0382 |     0.0005 |    Dislike |    Dislike
Opor Ayam              |     0.9408 |     0.9406 |     0.0002 |       Like |       Like
------------------------------------------------------------------------------------------
Mean Absolute Error:    0.000144
Maximum Absolute Error: 0.000493
